# Notebook 2 - Análise de Negócio do e-Commerce

Este notebook reúne uma análise exploratória e diagnóstica da operação de e-commerce, com foco em **crescimento**, **receita**, **logística**, **pagamentos**, **retenção** e **satisfação do cliente**.  
A proposta é transformar a base tratada em insumos claros para tomada de decisão, combinando tabelas de apoio, indicadores e visualizações.

## 1. Preparação do ambiente e carregamento da base

Nesta primeira etapa, são importadas as bibliotecas, carregada a base principal e definidas funções auxiliares de formatação.  
O objetivo é garantir uma base consistente para as análises seguintes e melhorar a legibilidade das saídas.

In [0]:
# Importação das bibliotecas usadas em manipulação de dados, visualização e análise exploratória.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns


In [0]:
# Leitura da base tratada que servirá como fonte principal para todas as análises do notebook.
df_delivered = pd.read_csv("data/Inputs/base_tratada.csv")


In [0]:
# Visualização inicial para validar a estrutura da base e conferir se o carregamento ocorreu corretamente.
df_delivered.head(5)

In [0]:
# Funções auxiliares para formatar valores monetários e inteiros de forma mais legível nas tabelas e gráficos.
def formatar_brl(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def formatar_int(valor):
    return f"{valor:,}".replace(",", ".")

## 2. Visão geral do negócio

A seção a seguir consolida os principais indicadores executivos da operação e apresenta recortes por tempo, categoria, estado, seller e forma de pagamento.  
Ela responde perguntas como: **o negócio está crescendo?**, **onde a receita está concentrada?** e **quais segmentos merecem prioridade?**

In [0]:
# Cálculo dos KPIs executivos centrais do negócio: volume, base de clientes, sellers, receita, ticket e satisfação média.
kpis = {
    "total_pedidos": df_delivered["order_id"].nunique(),
    "total_clientes": df_delivered["customer_unique_id"].nunique(),
    "total_sellers": df_delivered["seller_id"].nunique(),
    "receita_total": df_delivered["item_total"].sum(),
    "ticket_medio": df_delivered.groupby("order_id")["item_total"].sum().mean(),
    "review_medio": df_delivered["review_score_mean"].mean()
}
kpis


In [0]:
#evo mensal
monthly_orders = (
    df_delivered
    .groupby("purchase_year_month", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("item_total", "sum")
    )
    .sort_values("purchase_year_month")
)


### 2.1 KPIs executivos

Os KPIs abaixo oferecem uma visão sintética da operação. Eles servem como ponto de partida para interpretar volume, monetização e experiência do cliente.

In [0]:
#RECEITA POR CATEGORI   
revenue_by_category = (
    df_delivered
    .groupby("product_category", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("item_total", "sum"),
        avg_review=("review_score_mean", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)


In [0]:
# Resumo da receita por estado do cliente, útil para leitura regional da demanda.
revenue_by_state = (
    df_delivered
    .groupby("customer_state", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("item_total", "sum"),
        avg_review=("review_score_mean", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)


In [0]:
# Agrupamento por seller para avaliar receita, volume, satisfação e taxa de atraso.
seller_performance = (
    df_delivered
    .groupby("seller_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("item_total", "sum"),
        avg_review=("review_score_mean", "mean"),
        delay_rate=("is_delayed", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)


In [0]:
# Análise consolidada por meio de pagamento, combinando pedidos, receita, ticket e parcelamento médio.
payment_analysis = (
    df_delivered
    .groupby("payment_type_main", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("item_total", "sum"),
        avg_ticket=("item_total", "mean"),
        avg_installments=("payment_installments_max", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)


In [0]:
# Gráfico de linha da evolução mensal de pedidos para identificar tendência, sazonalidade e pontos de inflexão.
plt.figure(figsize=(10,6))

x = monthly_orders["purchase_year_month"]
y = monthly_orders["total_orders"]

plt.plot(
    x,
    y,
    marker="o",
    linewidth=2
)

plt.fill_between(
    x,
    y,
    alpha=0.15
)

plt.title("Pedidos por mês")
plt.xlabel("Mês")
plt.ylabel("Pedidos")

plt.grid(alpha=0.3)

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()


In [0]:
# Gráfico de linha da evolução mensal da receita para comparar crescimento financeiro ao longo do tempo.


plt.figure(figsize=(10,6))

x = monthly_orders["purchase_year_month"]
y = monthly_orders["total_revenue"]

cor = "#2E86C1"

plt.plot(
    x,
    y,
    marker="o",
    linewidth=2,
    color=cor
)

plt.fill_between(
    x,
    y,
    color=cor,
    alpha=0.15
)

plt.title("Receita mensal")
plt.xlabel("Mês")
plt.ylabel("Receita")

ax = plt.gca()

ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: formatar_brl(x))
)

plt.grid(alpha=0.3)

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()



### 2.2 Crescimento, receita e concentração

Nesta subseção, os indicadores são desdobrados em perspectivas temporais e segmentadas. A leitura conjunta ajuda a identificar sazonalidade, concentração de receita e desempenho dos principais grupos de negócio.

In [0]:
# Preparação do recorte de categorias líderes em receita para visualização comparativa.
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

top_cat = (
    revenue_by_category
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

plt.figure(figsize=(11,6))

ax = plt.gca()

ax.bar(
    top_cat["product_category"],
    top_cat["total_revenue"],
    color="#2E86C1"
)

ax.set_title("Top 10 categorias por receita")
ax.set_xlabel("Categoria")
ax.set_ylabel("Receita (R$)")

# formatação monetária brasileira
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: formatar_brl(x))
)

# rótulos levemente inclinados
plt.xticks(rotation=30, ha="right")

# grid leve
ax.grid(axis="y", alpha=0.3)

# valores acima das barras
for i, v in enumerate(top_cat["total_revenue"]):
    ax.text(
        i,
        v,
        formatar_brl(v),
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()

plt.show()


In [0]:
# Gráfico horizontal com os estados de maior receita, facilitando comparação entre regiões.
top_states = revenue_by_state.head(10).sort_values("total_revenue")

plt.figure(figsize=(10,8))

plt.barh(
    top_states["customer_state"],
    top_states["total_revenue"]
)

plt.title("Top estados por receita")
plt.xlabel("Receita")
plt.ylabel("Estado")

ax = plt.gca()

ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: formatar_brl(x))
)

for i, v in enumerate(top_states["total_revenue"]):
    plt.text(v, i, formatar_brl(v), va="center")

plt.grid(axis="x", alpha=0.3)

plt.show()


In [0]:
# Visualização da distribuição de pedidos por tipo principal de pagamento.
plt.figure(figsize=(8,6))

plt.bar(
    payment_analysis["payment_type_main"],
    payment_analysis["total_revenue"]
)

plt.title("Receita por meio de pagamento")
plt.xlabel("Tipo de pagamento")
plt.ylabel("Receita")

ax = plt.gca()

ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: formatar_brl(x))
)

for i, v in enumerate(payment_analysis["total_revenue"]):
    plt.text(i, v, formatar_brl(v), ha="center", va="bottom")

plt.grid(axis="y", alpha=0.3)

plt.show()


In [0]:
# Transformação do dicionário de KPIs em DataFrame para exibição tabular.
kpis_df = pd.DataFrame(
    list(kpis.items()),
    columns=["Indicador", "Valor"]
)

In [0]:
# Formatação dos indicadores para exibição final, diferenciando métricas financeiras de contagens.
kpis_df["Valor"] = kpis_df.apply(
    lambda row: formatar_brl(row["Valor"]) 
    if row["Indicador"] in ["receita_total", "ticket_medio"]
    else formatar_int(row["Valor"]) 
    if row["Indicador"] in ["total_pedidos", "total_clientes", "total_sellers"]
    else f"{row['Valor']:.2f}",
    axis=1
)

In [0]:
# Renomeação técnica dos indicadores.
nomes = {
    "total_pedidos": "Total de pedidos",
    "total_clientes": "Total de clientes",
    "total_sellers": "Total de sellers",
    "receita_total": "Receita total",
    "ticket_medio": "Ticket médio",
    "review_medio": "Avaliação média"
}

kpis_df["Indicador"] = kpis_df["Indicador"].map(nomes)


In [0]:
# Exibição da tabela consolidada de KPIs do negócio.
display(kpis_df)

In [0]:
# Cálculo do crescimento mensal com pedidos, receita e ticket médio por período.
crescimento_mensal = (
    df_delivered
    .groupby("purchase_year_month", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        receita=("item_total", "sum")
    )
)

crescimento_mensal["ticket_medio"] = (
    crescimento_mensal["receita"] /
    crescimento_mensal["pedidos"]
)

crescimento_mensal

In [0]:
# Gráfico com eixo duplo para acompanhar simultaneamente volume de pedidos e receita mensal.
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, ax1 = plt.subplots(figsize=(11,6))

ax1.plot(
    crescimento_mensal["purchase_year_month"],
    crescimento_mensal["pedidos"],
    marker="o",
    label="Pedidos"
)

ax1.set_ylabel("Pedidos")
ax1.set_xlabel("Mês")
ax1.grid(alpha=0.3)

plt.xticks(rotation=30)

ax2 = ax1.twinx()

ax2.plot(
    crescimento_mensal["purchase_year_month"],
    crescimento_mensal["receita"],
    marker="o",
    color="green",
    label="Receita"
)

ax2.set_ylabel("Receita (R$)")
ax2.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: formatar_brl(x))
)

plt.title("Evolução mensal de pedidos e receita")

plt.tight_layout()
plt.show()

In [0]:
# Visualização isolada do ticket médio mensal para observar comportamento de monetização por pedido.
plt.figure(figsize=(10,6))

plt.plot(
    crescimento_mensal["purchase_year_month"],
    crescimento_mensal["ticket_medio"],
    marker="o",
    color="#E67E22"
)

plt.fill_between(
    crescimento_mensal["purchase_year_month"],
    crescimento_mensal["ticket_medio"],
    alpha=0.15,
    color="#E67E22"
)

plt.title("Ticket médio mensal")
plt.xlabel("Mês")
plt.ylabel("Ticket médio")

ax = plt.gca()

ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: formatar_brl(x))
)

plt.grid(alpha=0.3)
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

In [0]:
# Tabela de receita por categoria para identificar concentração e relevância comercial.
categoria_receita = (
    df_delivered
    .groupby("product_category", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        receita=("item_total", "sum")
    )
    .sort_values("receita", ascending=False)
)

In [0]:
# Gráfico das categorias com maior participação em receita.
top_cat = categoria_receita.head(10)

plt.figure(figsize=(11,6))

plt.bar(
    top_cat["product_category"],
    top_cat["receita"],
    color="#2E86C1"
)

plt.title("Top 10 categorias por receita")
plt.xlabel("Categoria")
plt.ylabel("Receita")

ax = plt.gca()

ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: formatar_brl(x))
)

plt.xticks(rotation=30, ha="right")

plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# Tabela de receita por estado para avaliar concentração geográfica.
receita_estado = (
    df_delivered
    .groupby("customer_state", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        receita=("item_total", "sum")
    )
    .sort_values("receita", ascending=False)
)

In [0]:
# Gráfico dos estados com maior geração de receita.
top_states = receita_estado.head(10)

plt.figure(figsize=(10,6))

plt.bar(
    top_states["customer_state"],
    top_states["receita"],
    color="#27AE60"
)

plt.title("Top estados por receita")
plt.xlabel("Estado")
plt.ylabel("Receita")

ax = plt.gca()

ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: formatar_brl(x))
)

plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# Ranking de produtos por pedidos e receita para identificar itens mais relevantes comercialmente.
top_produtos = (
    df_delivered
    .groupby("product_id", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        receita=("item_total", "sum")
    )
    .sort_values("receita", ascending=False)
    .head(10)
)

top_produtos

In [0]:
# Ranking de sellers por volume e receita para identificar parceiros de maior impacto.
top_sellers = (
    df_delivered
    .groupby("seller_id", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        receita=("item_total", "sum"),
        review_medio=("review_score_mean", "mean")
    )
    .sort_values("receita", ascending=False)
    .head(10)
)

top_sellers

In [0]:
# Visualização dos sellers com maior receita.
plt.figure(figsize=(10,6))

plt.bar(
    top_sellers["seller_id"],
    top_sellers["receita"],
    color="#8E44AD"
)

plt.title("Top sellers por receita")
plt.xlabel("Seller")
plt.ylabel("Receita")

ax = plt.gca()

ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: formatar_brl(x))
)

plt.xticks(rotation=45)

plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# Preparação da base agregada para relacionar pedidos e receita por categoria em um gráfico combinado.
produtos = (
    df_delivered
    .groupby("product_category", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        receita=("item_total", "sum")
    )
)

plt.figure(figsize=(10,6))

plt.scatter(
    produtos["pedidos"],
    produtos["receita"],
    alpha=0.6
)

plt.xlabel("Quantidade de pedidos")
plt.ylabel("Receita")

ax = plt.gca()

ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: formatar_brl(x))
)

plt.title("Receita vs quantidade de pedidos por categoria")

plt.grid(alpha=0.3)

plt.show()

In [0]:
# Gráfico com eixo duplo para comparar, por categoria, volume de pedidos e receita total.
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, ax1 = plt.subplots(figsize=(11,6))

# Linha de pedidos
ax1.plot(
    crescimento_mensal["purchase_year_month"],
    crescimento_mensal["pedidos"],
    marker="o",
    label="Pedidos"
)

ax1.set_ylabel("Pedidos")
ax1.set_xlabel("Mês")
ax1.grid(alpha=0.3)

plt.xticks(rotation=30)

# Segundo eixo (receita)
ax2 = ax1.twinx()

ax2.plot(
    crescimento_mensal["purchase_year_month"],
    crescimento_mensal["receita"],
    marker="o",
    color="green",
    label="Receita"
)

ax2.set_ylabel("Receita (R$)")
ax2.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: formatar_brl(x))
)

# Título
plt.title("Evolução mensal de pedidos e receita")

# ===== LEGENDA CONSOLIDADA =====
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines_1 + lines_2,
    labels_1 + labels_2,
    loc="upper left",
    frameon=True
)

plt.tight_layout()
plt.show()

## 3. Análise de logística e lead time

Nesta seção, o objetivo é medir a eficiência operacional da jornada de entrega. São calculados os tempos entre compra, aprovação, postagem e entrega, além do percentual de pedidos atrasados e sua relação com a satisfação do cliente.

In [0]:
# =========================
# 2. Verificar colunas
# =========================
print("Colunas da base:")
print(df_delivered.columns.tolist())

In [0]:
# =========================
# 3. Converter datas
# =========================
colunas_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in colunas_data:
    if col in df_delivered.columns:
        df_delivered[col] = pd.to_datetime(df_delivered[col], errors="coerce")

In [0]:
# =========================
# 4. Criar lead times
# =========================
df_log = df_delivered.copy()

df_log["lt_compra_aprovacao"] = (
    df_log["order_approved_at"] - df_log["order_purchase_timestamp"]
).dt.total_seconds() / 86400

df_log["lt_aprovacao_postagem"] = (
    df_log["order_delivered_carrier_date"] - df_log["order_approved_at"]
).dt.total_seconds() / 86400

df_log["lt_postagem_entrega"] = (
    df_log["order_delivered_customer_date"] - df_log["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

df_log["lt_total"] = (
    df_log["order_delivered_customer_date"] - df_log["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [0]:
# =========================
# 5. Criar atraso de entrega
# =========================
df_log["dias_atraso"] = (
    df_log["order_delivered_customer_date"] - df_log["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

df_log["pedido_atrasado"] = df_log["dias_atraso"] > 0

In [0]:
# =========================
# 6. Identificar coluna de review
# =========================
col_review = None

for c in ["review_score_mean", "review_score"]:
    if c in df_log.columns:
        col_review = c
        break

print(f"Coluna de review identificada: {col_review}")


In [0]:
# =========================
# 7. Tabela KPI logística
# =========================
kpis_logistica = pd.DataFrame({
    "Indicador": [
        "Lead time compra → aprovação",
        "Lead time aprovação → postagem",
        "Lead time postagem → entrega",
        "Lead time total compra → entrega",
        "% pedidos atrasados"
    ],
    "Valor": [
        df_log["lt_compra_aprovacao"].mean(),
        df_log["lt_aprovacao_postagem"].mean(),
        df_log["lt_postagem_entrega"].mean(),
        df_log["lt_total"].mean(),
        df_log["pedido_atrasado"].mean() * 100
    ]
})

kpis_logistica["Valor"] = kpis_logistica["Valor"].round(2)
print("\nKPIs de Logística:")
display(kpis_logistica)

In [0]:
# =========================
# 8. Resumo estatístico
# =========================
resumo_leadtime = pd.DataFrame({
    "Etapa": [
        "Compra → Aprovação",
        "Aprovação → Postagem",
        "Postagem → Entrega",
        "Total"
    ],
    "Média (dias)": [
        df_log["lt_compra_aprovacao"].mean(),
        df_log["lt_aprovacao_postagem"].mean(),
        df_log["lt_postagem_entrega"].mean(),
        df_log["lt_total"].mean()
    ],
    "Mediana (dias)": [
        df_log["lt_compra_aprovacao"].median(),
        df_log["lt_aprovacao_postagem"].median(),
        df_log["lt_postagem_entrega"].median(),
        df_log["lt_total"].median()
    ],
    "P95 (dias)": [
        df_log["lt_compra_aprovacao"].quantile(0.95),
        df_log["lt_aprovacao_postagem"].quantile(0.95),
        df_log["lt_postagem_entrega"].quantile(0.95),
        df_log["lt_total"].quantile(0.95)
    ]
}).round(2)

print("\nResumo estatístico dos lead times:")
display(resumo_leadtime)

In [0]:
# =========================
# 9. Gráfico de lead time médio por etapa
# =========================
plt.figure(figsize=(10, 5))

etapas = ["Compra→Aprovação", "Aprovação→Postagem", "Postagem→Entrega", "Total"]
valores = [
    df_log["lt_compra_aprovacao"].mean(),
    df_log["lt_aprovacao_postagem"].mean(),
    df_log["lt_postagem_entrega"].mean(),
    df_log["lt_total"].mean()
]

plt.bar(etapas, valores)
plt.title("Lead Time Médio por Etapa")
plt.ylabel("Dias")
plt.grid(axis="y", alpha=0.3)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [0]:
# =========================
# 10. Boxplot dos lead times
# =========================
df_box = df_log[[
    "lt_compra_aprovacao",
    "lt_aprovacao_postagem",
    "lt_postagem_entrega",
    "lt_total"
]].copy()

df_box = df_box.melt(var_name="Etapa", value_name="Dias")
df_box["Etapa"] = df_box["Etapa"].replace({
    "lt_compra_aprovacao": "Compra→Aprovação",
    "lt_aprovacao_postagem": "Aprovação→Postagem",
    "lt_postagem_entrega": "Postagem→Entrega",
    "lt_total": "Total"
})

plt.figure(figsize=(11, 6))
sns.boxplot(data=df_box, x="Etapa", y="Dias")
plt.title("Distribuição dos Lead Times")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
# =========================
# 11. Histograma do lead time total
# =========================
plt.figure(figsize=(10, 5))
plt.hist(df_log["lt_total"].dropna(), bins=30)
plt.title("Distribuição do Lead Time Total")
plt.xlabel("Dias")
plt.ylabel("Frequência")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
# =========================
# 12. Correlação atraso x review
# =========================
if col_review is not None:
    df_corr = df_log[["dias_atraso", "pedido_atrasado", col_review]].dropna().copy()

    correlacao = df_corr["dias_atraso"].corr(df_corr[col_review])
    print(f"\nCorrelação entre dias de atraso e review score: {correlacao:.2f}")

    resumo_review = (
        df_corr.groupby("pedido_atrasado", as_index=False)
        .agg(
            pedidos=(col_review, "count"),
            review_medio=(col_review, "mean"),
            atraso_medio=("dias_atraso", "mean")
        )
        .round(2)
    )

    resumo_review["pedido_atrasado"] = resumo_review["pedido_atrasado"].replace({
        False: "No prazo",
        True: "Atrasado"
    })

    print("\nResumo review por atraso:")
    display(resumo_review)

    # Scatter plot
    plt.figure(figsize=(10, 5))
    plt.scatter(df_corr["dias_atraso"], df_corr[col_review], alpha=0.4)
    plt.title("Correlação entre Dias de Atraso e Review Score")
    plt.xlabel("Dias de atraso")
    plt.ylabel("Review score")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Barras no prazo vs atrasado
    plt.figure(figsize=(7, 5))
    plt.bar(resumo_review["pedido_atrasado"], resumo_review["review_medio"])
    plt.title("Review Médio: Pedidos no Prazo vs Atrasados")
    plt.ylabel("Review médio")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Faixas de atraso
    bins = [-1000, 0, 3, 7, 15, 1000]
    labels = ["No prazo", "1-3 dias", "4-7 dias", "8-15 dias", "15+ dias"]

    df_corr["faixa_atraso"] = pd.cut(df_corr["dias_atraso"], bins=bins, labels=labels)

    faixa_review = (
        df_corr.groupby("faixa_atraso", observed=False, as_index=False)
        .agg(
            pedidos=(col_review, "count"),
            review_medio=(col_review, "mean")
        )
        .round(2)
    )

    print("\nReview por faixa de atraso:")
    display(faixa_review)

    plt.figure(figsize=(9, 5))
    plt.bar(faixa_review["faixa_atraso"].astype(str), faixa_review["review_medio"])
    plt.title("Review Médio por Faixa de Atraso")
    plt.xlabel("Faixa de atraso")
    plt.ylabel("Review médio")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

else:
    print("\nNenhuma coluna de review encontrada na base.")

In [0]:
# =========================
# 13. Desempenho por região / estado
# =========================
col_estado = None
for c in ["customer_state", "seller_state"]:
    if c in df_log.columns:
        col_estado = c
        break

if col_estado is not None:
    agg_dict = {
        "order_id": pd.Series.nunique,
        "lt_total": "mean",
        "pedido_atrasado": "mean"
    }

    if col_review is not None:
        agg_dict[col_review] = "mean"

    desempenho_estado = (
        df_log.groupby(col_estado, as_index=False)
        .agg(agg_dict)
        .rename(columns={
            "order_id": "pedidos",
            "lt_total": "lead_time_medio",
            "pedido_atrasado": "pct_atraso",
            col_review if col_review else "": "review_medio"
        })
    )

    desempenho_estado["lead_time_medio"] = desempenho_estado["lead_time_medio"].round(2)
    desempenho_estado["pct_atraso"] = (desempenho_estado["pct_atraso"] * 100).round(2)

    if col_review is not None and "review_medio" in desempenho_estado.columns:
        desempenho_estado["review_medio"] = desempenho_estado["review_medio"].round(2)

    print(f"\nDesempenho por {col_estado}:")
    display(desempenho_estado.sort_values("pct_atraso", ascending=False))

    # Heatmap
    heat_cols = [col_estado, "lead_time_medio", "pct_atraso"]
    if col_review is not None and "review_medio" in desempenho_estado.columns:
        heat_cols.append("review_medio")

    heatmap_df = desempenho_estado[heat_cols].set_index(col_estado)

    plt.figure(figsize=(10, max(6, len(heatmap_df) * 0.4)))
    sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap="YlOrRd")
    plt.title(f"Mapa de Calor de Desempenho Logístico por {col_estado}")
    plt.tight_layout()
    plt.show()

else:
    print("\nNenhuma coluna de estado encontrada na base.")

## 4. Comportamento de pagamento, RFM e retenção

Aqui são analisados os meios de pagamento mais utilizados, o efeito do parcelamento sobre o ticket médio, o perfil de clientes por **Recência, Frequência e Monetário (RFM)** e a retenção por coorte.

In [0]:
# Cópia da base para a frente de análise de pagamentos, mantendo o dataframe original preservado.

df_pag = df_delivered.copy() 

In [0]:

# =========================
# 1.1 Distribuição de meios de pagamento
# =========================
pagamentos = (
    df_delivered.groupby("payment_type_main", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        receita=("payment_value_total", "sum")
    )
    .sort_values("pedidos", ascending=False)
)

pagamentos["pct_pedidos"] = (
    pagamentos["pedidos"] / pagamentos["pedidos"].sum() * 100
).round(2)

pagamentos["receita"] = pagamentos["receita"].round(2)

display(pagamentos)

In [0]:
# Análise de parcelamento para observar relação entre número de parcelas, pedidos e ticket médio.
parcelamento = (
    df_delivered.groupby("payment_installments_max", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        ticket_medio=("payment_value_total", "mean")
    )
    .sort_values("payment_installments_max")
)

parcelamento["ticket_medio"] = parcelamento["ticket_medio"].round(2)

display(parcelamento)

In [0]:


# Meios de pagamento
plt.figure(figsize=(8,5))
plt.bar(pagamentos["payment_type_main"], pagamentos["pedidos"])
plt.title("Distribuição de Meios de Pagamento")
plt.ylabel("Pedidos")
plt.grid(axis="y", alpha=0.3)
plt.xticks(rotation=20)
plt.show()



In [0]:
# Parcelas vs ticket
plt.figure(figsize=(8,5))
plt.plot(parcelamento["payment_installments_max"], parcelamento["ticket_medio"], marker="o")
plt.title("Parcelamento vs Ticket Médio")
plt.xlabel("Número de Parcelas")
plt.ylabel("Ticket Médio")
plt.grid(alpha=0.3)
plt.show()

In [0]:
# Data de referência (última data da base)
data_ref = df_delivered["order_purchase_timestamp"].max()

rfm = (
    df_delivered.groupby("customer_unique_id")
    .agg(
        recencia=("order_purchase_timestamp", lambda x: (data_ref - x.max()).days),
        frequencia=("order_id", "nunique"),
        monetario=("payment_value_total", "sum")
    )
    .reset_index()
)

rfm["monetario"] = rfm["monetario"].round(2)

display(rfm.head())

In [0]:
# Criar scores por quartil
rfm["R_score"] = pd.qcut(rfm["recencia"], 4, labels=[4,3,2,1])
rfm["F_score"] = pd.qcut(rfm["frequencia"].rank(method="first"), 4, labels=[1,2,3,4])
rfm["M_score"] = pd.qcut(rfm["monetario"], 4, labels=[1,2,3,4])

rfm["RFM_score"] = (
    rfm["R_score"].astype(str) +
    rfm["F_score"].astype(str) +
    rfm["M_score"].astype(str)
)

display(rfm.head())

In [0]:
# Regras de segmentação RFM para traduzir os scores em grupos de negócio acionáveis.
def segmentar(row):
    if row["RFM_score"] == "444":
        return "Melhores clientes"
    elif row["R_score"] == 4:
        return "Clientes recentes"
    elif row["F_score"] == 4:
        return "Clientes frequentes"
    elif row["M_score"] == 4:
        return "Alto valor"
    else:
        return "Outros"

rfm["segmento"] = rfm.apply(segmentar, axis=1)

segmentos = (
    rfm.groupby("segmento", as_index=False)
    .agg(clientes=("customer_unique_id", "count"))
    .sort_values("clientes", ascending=False)
)

display(segmentos)

In [0]:
# Construção da base de coorte para medir retenção a partir do mês da primeira compra.
df_cohort = df_delivered.copy()

df_cohort["order_month"] = df_cohort["order_purchase_timestamp"].dt.to_period("M")
df_cohort["cohort_month"] = (
    df_cohort.groupby("customer_unique_id")["order_month"]
    .transform("min")
)

# diferença em meses
df_cohort["cohort_index"] = (
    (df_cohort["order_month"] - df_cohort["cohort_month"])
    .apply(lambda x: x.n)
)

In [0]:
# Tabela matricial de retenção por coorte, base para o heatmap de recompra.
cohort_table = (
    df_cohort.groupby(["cohort_month", "cohort_index"])
    .agg(clientes=("customer_unique_id", "nunique"))
    .reset_index()
)

cohort_pivot = cohort_table.pivot(
    index="cohort_month",
    columns="cohort_index",
    values="clientes"
)

# Normalizar
cohort_size = cohort_pivot.iloc[:, 0]
cohort_retencao = cohort_pivot.divide(cohort_size, axis=0)

display(cohort_retencao.head())

In [0]:
# Heatmap de retenção para leitura visual do comportamento de recompra ao longo dos meses.
import seaborn as sns

plt.figure(figsize=(12,8))
sns.heatmap(cohort_retencao, annot=True, fmt=".2f", cmap="Blues")
plt.title("Retenção de Clientes por Coorte")
plt.xlabel("Meses desde a primeira compra")
plt.ylabel("Cohort (mês de aquisição)")
plt.show()

## 5. Satisfação do cliente e drivers de experiência

Nesta etapa, avaliamos como prazo de entrega, faixa de preço e categoria de produto se relacionam com a nota média de avaliação. O foco é entender os principais fatores associados à satisfação e ao risco de churn.

In [0]:
# Correlação inicial entre tempo de entrega e nota de avaliação do cliente.
df_driver = df_delivered.copy()

corr_entrega = df_driver["delivery_time_days"].corr(df_driver[col_review])
print(f"Correlação: tempo de entrega vs review = {corr_entrega:.2f}")

In [0]:
# Criação de faixas de prazo de entrega para comparar satisfação média entre grupos.
bins = [0, 3, 7, 15, 30, 100]
labels = ["0-3", "4-7", "8-15", "16-30", "30+"]

df_driver["faixa_entrega"] = pd.cut(
    df_driver["delivery_time_days"],
    bins=bins,
    labels=labels
)

entrega_review = (
    df_driver.groupby("faixa_entrega", observed=False)
    .agg(review_medio=(col_review, "mean"))
    .reset_index()
)

display(entrega_review)

In [0]:
# Gráfico da nota média por faixa de entrega, destacando o efeito do prazo sobre a experiência.
plt.figure(figsize=(8,5))
plt.bar(entrega_review["faixa_entrega"].astype(str), entrega_review["review_medio"])
plt.title("Review Médio por Tempo de Entrega")
plt.xlabel("Dias de entrega")
plt.ylabel("Review médio")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [0]:
# Correlação entre valor do pedido e avaliação média.
corr_preco = df_driver["item_total"].corr(df_driver[col_review])
print(f"Correlação: preço vs review = {corr_preco:.2f}")

In [0]:
# Criação de faixas de preço para avaliar como ticket se relaciona com satisfação.
df_driver["faixa_preco"] = pd.qcut(df_driver["item_total"], 5)

preco_review = (
    df_driver.groupby("faixa_preco")
    .agg(review_medio=(col_review, "mean"))
    .reset_index()
)

display(preco_review)

In [0]:
# Repetição da análise por faixa de preço para gerar a tabela consolidada usada na visualização.
df_driver["faixa_preco"] = pd.qcut(df_driver["item_total"], 5)

preco_review = (
    df_driver.groupby("faixa_preco")
    .agg(review_medio=(col_review, "mean"))
    .reset_index()
)

display(preco_review)

In [0]:
# Ranking das categorias com pior avaliação média, considerando apenas volume relevante de pedidos.
categoria_review = (
    df_driver.groupby("product_category", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        review_medio=(col_review, "mean")
    )
    .query("pedidos > 50")
    .sort_values("review_medio")
)

display(categoria_review.head(10))

In [0]:
# Preparação da base para risco de churn a partir de sinais de insatisfação.
df_churn = df_delivered.copy()

df_churn["cliente_insatisfeito"] = df_churn[col_review] <= 2

In [0]:
# Consolidação dos indicadores por cliente para medir recorrência, atraso e satisfação média.
churn_cliente = (
    df_churn.groupby("customer_unique_id")
    .agg(
        pedidos=("order_id", "nunique"),
        review_medio=(col_review, "mean"),
        atraso_medio=("delay_days", "mean"),
        pct_atraso=("is_delayed", "mean")
    )
    .reset_index()
)

churn_cliente["pct_atraso"] = churn_cliente["pct_atraso"] * 100

In [0]:
# Regras simples para classificar o risco de churn com base em satisfação e atraso.
def risco(row):
    if row["review_medio"] <= 2:
        return "Alto risco"
    elif row["pct_atraso"] > 30:
        return "Médio risco"
    elif row["pedidos"] == 1:
        return "Baixo engajamento"
    else:
        return "Saudável"

churn_cliente["risco_churn"] = churn_cliente.apply(risco, axis=1)

display(churn_cliente.head())

In [0]:
# Distribuição da base de clientes por nível estimado de risco de churn.
risco_dist = (
    churn_cliente.groupby("risco_churn", as_index=False)
    .agg(clientes=("customer_unique_id", "count"))
    .sort_values("clientes", ascending=False)
)

display(risco_dist)

## 6. Oportunidades de negócio e recomendações operacionais

A última parte transforma os achados analíticos em direcionamentos práticos. São priorizadas rotas críticas, categorias com espaço para estratégia de pricing/cross-sell e sellers que combinam boa experiência com eficiência operacional.

In [0]:
# Mapeamento das rotas seller-estado do cliente para priorização logística.
rotas = (
    df_delivered.groupby(["seller_state", "customer_state"], as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        frete_medio=("freight_value", "mean"),
        tempo_entrega=("delivery_time_days", "mean"),
        atraso_pct=("is_delayed", "mean")
    )
)

rotas["atraso_pct"] = rotas["atraso_pct"] * 100

# filtrar rotas relevantes
rotas_criticas = (
    rotas
    .query("pedidos > 50")
    .sort_values(["atraso_pct", "tempo_entrega"], ascending=False)
)

display(rotas_criticas.head(10))

In [0]:
# Cálculo do peso do frete nas rotas críticas em relação ao ticket médio global.
rotas_criticas["frete_pct"] = (
    rotas_criticas["frete_medio"] /
    df_delivered["item_total"].mean()
) * 100

In [0]:
# Análise por categoria para discutir pricing, mix e oportunidades de cross-sell.
pricing_categoria = (
    df_delivered.groupby("product_category", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        receita=("item_total", "sum"),
        ticket_medio=("item_total", "mean"),
        review_medio=("review_score_mean", "mean")
    )
    .query("pedidos > 50")
    .sort_values("receita", ascending=False)
)

display(pricing_categoria.head(10))

In [0]:
# Correlação global entre preço e satisfação para verificar sensibilidade da experiência ao valor cobrado.
corr_preco_review = df_delivered["item_total"].corr(df_delivered["review_score_mean"])
print(f"Correlação preço vs satisfação: {corr_preco_review:.2f}")

In [0]:
# Cálculo da quantidade de itens por pedido para apoiar hipóteses de cross-sell e composição de cesta.
itens_pedido = (
    df_delivered.groupby("order_id")
    .agg(itens=("order_item_id", "count"))
)

cross_sell = (
    itens_pedido["itens"]
    .value_counts()
    .sort_index()
)

display(cross_sell)

In [0]:
# Consolidação do desempenho de sellers com foco em volume, receita, satisfação e logística.
sellers = (
    df_delivered.groupby("seller_id", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        receita=("item_total", "sum"),
        review_medio=("review_score_mean", "mean"),
        tempo_entrega=("delivery_time_days", "mean"),
        atraso_pct=("is_delayed", "mean")
    )
)

sellers["atraso_pct"] = sellers["atraso_pct"] * 100

# filtrar sellers relevantes
sellers = sellers.query("pedidos > 50")

display(sellers.head())

In [0]:
# Classificação dos sellers em perfis de performance para facilitar priorização gerencial.
def classificar(row):
    if row["review_medio"] >= 4.5 and row["atraso_pct"] < 10:
        return "Top performer"
    elif row["review_medio"] < 3:
        return "Crítico"
    elif row["atraso_pct"] > 30:
        return "Logística ruim"
    else:
        return "Regular"

sellers["classificacao"] = sellers.apply(classificar, axis=1)

ranking = (
    sellers.groupby("classificacao", as_index=False)
    .agg(qtd_sellers=("seller_id", "count"))
)

display(ranking)

# Testando algumas novas Analises tentando fazer a correlação entre o crescimento/receita com a logista 

In [0]:
# garantir datetime
df_delivered["order_purchase_timestamp"] = pd.to_datetime(
    df_delivered["order_purchase_timestamp"]
)

# filtro a partir de Dez/2016
df_filtrado = df_delivered[
    df_delivered["order_purchase_timestamp"] >= "2016-12-01"
].copy()

In [0]:
crescimento_sla = (
    df_filtrado.groupby("purchase_year_month", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        receita=("item_total", "sum"),
        tempo_entrega=("delivery_time_days", "mean"),
        atraso_pct=("is_delayed", "mean")
    )
)

crescimento_sla["atraso_pct"] = crescimento_sla["atraso_pct"] * 100

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, ax1 = plt.subplots(figsize=(11,5))

# Pedidos → azul
ax1.plot(
    crescimento_sla["purchase_year_month"],
    crescimento_sla["pedidos"],
    color="#1f77b4",
    marker="o",
    label="Pedidos"
)

ax1.set_ylabel("Pedidos", color="#1f77b4")
ax1.tick_params(axis='y', labelcolor="#1f77b4")
ax1.set_xlabel("Mês")
ax1.grid(alpha=0.3)
plt.xticks(rotation=30)

# Receita → verde
ax2 = ax1.twinx()

ax2.plot(
    crescimento_sla["purchase_year_month"],
    crescimento_sla["receita"],
    color="#2ca02c",
    linestyle="--",
    marker="o",
    label="Receita"
)

ax2.set_ylabel("Receita (R$)", color="#2ca02c")
ax2.tick_params(axis='y', labelcolor="#2ca02c")

# formatação moeda
ax2.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: f"R$ {x:,.0f}".replace(",", "."))
)

# legenda
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

plt.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Crescimento de Pedidos e Receita (a partir de Dez/2016)")
plt.tight_layout()
plt.show()

In [0]:
fig, ax1 = plt.subplots(figsize=(11,5))

# Tempo entrega → laranja
ax1.plot(
    crescimento_sla["purchase_year_month"],
    crescimento_sla["tempo_entrega"],
    color="#ff7f0e",
    marker="o",
    label="Tempo Entrega (dias)"
)

ax1.set_ylabel("Dias", color="#ff7f0e")
ax1.tick_params(axis='y', labelcolor="#ff7f0e")
ax1.set_xlabel("Mês")
ax1.grid(alpha=0.3)
plt.xticks(rotation=30)

# Atraso → vermelho
ax2 = ax1.twinx()

ax2.plot(
    crescimento_sla["purchase_year_month"],
    crescimento_sla["atraso_pct"],
    color="#d62728",
    linestyle="--",
    marker="o",
    label="% Atraso"
)

ax2.set_ylabel("% Atraso", color="#d62728")
ax2.tick_params(axis='y', labelcolor="#d62728")

# legenda
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

plt.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Performance Logística (SLA) - a partir de Dez/2016")
plt.tight_layout()
plt.show()

In [0]:


df = df_delivered.copy()

# =========================
# 1. Ajustes básicos
# =========================
df["is_delayed"] = df["is_delayed"].astype(int)

# classificar pedidos
df["status_entrega"] = df["is_delayed"].map({
    1: "Atrasado",
    0: "No prazo"
})

# =========================
# 2. Review médio por status
# =========================
review_por_status = (
    df.groupby("status_entrega", as_index=False)
    .agg(review_medio=("review_score_mean", "mean"))
)

# =========================
# 3. Construir visão por cliente
# =========================
cliente_base = (
    df.groupby("customer_unique_id", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        receita_total=("payment_value_total", "sum"),
        review_medio=("review_score_mean", "mean"),
        atraso_medio=("is_delayed", "mean")
    )
)

# cliente teve pelo menos um atraso?
cliente_base["teve_atraso"] = np.where(cliente_base["atraso_medio"] > 0, "Teve atraso", "Sem atraso")

# recompra
cliente_base["recomprou"] = np.where(cliente_base["pedidos"] > 1, 1, 0)

# =========================
# 4. Métricas comparativas
# =========================
comparativo = (
    cliente_base.groupby("teve_atraso", as_index=False)
    .agg(
        clientes=("customer_unique_id", "count"),
        taxa_recompra=("recomprou", "mean"),
        receita_media_cliente=("receita_total", "mean"),
        review_medio_cliente=("review_medio", "mean")
    )
)

comparativo["taxa_recompra"] = comparativo["taxa_recompra"] * 100

# puxar review médio dos pedidos
review_dict = dict(zip(review_por_status["status_entrega"], review_por_status["review_medio"]))

# garantir ordem
ordem = ["Sem atraso", "Teve atraso"]
comparativo["teve_atraso"] = pd.Categorical(comparativo["teve_atraso"], categories=ordem, ordered=True)
comparativo = comparativo.sort_values("teve_atraso")

# review por pedido alinhado à lógica cliente
review_plot = [
    review_dict.get("No prazo", np.nan),
    review_dict.get("Atrasado", np.nan)
]

recompra_plot = comparativo["taxa_recompra"].tolist()
receita_plot = comparativo["receita_media_cliente"].tolist()

# % clientes com e sem atraso
pct_clientes = (
    comparativo["clientes"] / comparativo["clientes"].sum() * 100
).tolist()

# =========================
# 5. Plot bonito e claro
# =========================
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Impacto do Atraso na Experiência, Recompra e Receita Futura", fontsize=16)

grupos = ["Sem atraso", "Teve atraso"]

# 1) Distribuição de clientes
axes[0, 0].bar(grupos, pct_clientes)
axes[0, 0].set_title("% de Clientes")
axes[0, 0].set_ylabel("Percentual")
axes[0, 0].grid(axis="y", alpha=0.3)

for i, v in enumerate(pct_clientes):
    axes[0, 0].text(i, v + 0.5, f"{v:.1f}%", ha="center")

# 2) Review médio
axes[0, 1].bar(grupos, review_plot)
axes[0, 1].set_title("Review Médio")
axes[0, 1].set_ylabel("Nota")
axes[0, 1].set_ylim(0, 5.5)
axes[0, 1].grid(axis="y", alpha=0.3)

for i, v in enumerate(review_plot):
    axes[0, 1].text(i, v + 0.05, f"{v:.2f}", ha="center")

# 3) Taxa de recompra
axes[1, 0].bar(grupos, recompra_plot)
axes[1, 0].set_title("Taxa de Recompra")
axes[1, 0].set_ylabel("Percentual")
axes[1, 0].grid(axis="y", alpha=0.3)

for i, v in enumerate(recompra_plot):
    axes[1, 0].text(i, v + 0.5, f"{v:.1f}%", ha="center")

# 4) Receita média por cliente
axes[1, 1].bar(grupos, receita_plot)
axes[1, 1].set_title("Receita Média por Cliente")
axes[1, 1].set_ylabel("R$")
axes[1, 1].yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, pos: f"R$ {x:,.0f}".replace(",", "."))
)
axes[1, 1].grid(axis="y", alpha=0.3)

for i, v in enumerate(receita_plot):
    axes[1, 1].text(i, v + max(receita_plot)*0.02, f"R$ {v:,.0f}".replace(",", "."), ha="center")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()